In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

### Linear Regression model
#### Predict marks based on study hours

In [12]:
#x = clues =  hours of studies
x = np.array([[1], [2], [3],[4], [5],[6], [7],[8],[9],[10]])
#y= answer = marks
y = [20, 28, 35, 45, 52, 60, 68, 75, 85, 95]

#get the model
model = LinearRegression()

#train on 80% data and took test on 20%(0.2) data
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

# TEACH THE MODEL (FIT)
model.fit(x_train, y_train)
print("X test: ",x_test)
print("Y test: ",y_test)

predict = model.predict(x_test)

print("predict: ", predict)
print("for 4.5 hours of study: ",model.predict([[4.5]]))




X test:  [[9]
 [2]]
Y test:  [85, 28]
predict:  [84.97413793 27.52586207]
for 4.5 hours of study:  [48.04310345]


In [13]:
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import accuracy_score

In [15]:
# 1. OUR NEW STUDY MATERIALS
# x1 (Clues) = [Hours Studied, Classes Attended] -> Notice the 2 columns!
x1 = np.array([
    [1, 2], [2, 3], [3, 4], [4, 2],  # These students studied little & skipped class
    [6, 8], [7, 9], [8, 8], [9, 10]  # These students studied a lot & attended class
])

# y1 (Answers) = 0 for Fail, 1 for Pass
y1 = np.array([0, 0, 0, 0, 1, 1, 1, 1])

x_train, x_test, y_train, y_test = train_test_split(x1,y1, test_size=0.2, random_state=42)
print("x train: ",x_train)
print("x test: ", x_test)
print("y train: ", y_train)
print("y test: ", y_test)
print("Showing lucky shuffle problem")



x train:  [[ 1  2]
 [ 9 10]
 [ 3  4]
 [ 6  8]
 [ 4  2]
 [ 8  8]]
x test:  [[2 3]
 [7 9]]
y train:  [0 1 0 1 0 1]
y test:  [0 1]
Showing lucky shuffle problem


### Why Use Cross-Validation Instead of `train_test_split`?

When evaluating a machine learning model, relying on a simple `train_test_split` can be dangerous because of **The Lucky Shuffle Problem**. If the one test group happens to be incredibly easy (or hard) by pure chance, your model gets an inaccurate grade. 

**The Solution: Cross-Validation (`cross_val_score`)**
Instead of splitting the data once, it runs an automated loop where **every single chunk gets a turn to be the test set**. It is like giving a student 5 different exams over a semester and averaging their grades to find the honest truth.

---

### The Code: What changes?

When you upgrade to Cross-Validation, `scikit-learn` does all the heavy lifting for you in the background. You actually get to write *less* code.

#### Syntax We No Longer Need (The Old Way)
You get to completely delete these four manual steps:
1. `X_train, X_test... = train_test_split(...)` *(No more manual splitting)*
2. `model.fit(X_train, y_train)` *(No more manual studying)*
3. `predictions = model.predict(X_test)` *(No more manual testing)*
4. `accuracy_score(y_test, predictions)` *(No more manual grading)*

#### Syntax We Use Instead (The New Way)
We replace all of the above with just two new concepts:
1. **The Ultimate Grader:** `cross_val_score(model, X, y, cv=4)` 
   *(This single line automatically splits, fits, predicts, and scores the data 4 separate times).*
2. **The Average:** `scores.mean()` 
   *(Because the grader gives us a list of 4 separate scores, we use `.mean()` to calculate the final average).*

In [16]:
from sklearn.model_selection import cross_val_score

In [18]:
# a (Clues) = [Hours Studied, Classes Attended] -> Notice the 2 columns!
a = np.array([
    [1, 2], [2, 3], [3, 4], [4, 2],  # These students studied little & skipped class
    [6, 8], [7, 9], [8, 8], [9, 10]  # These students studied a lot & attended class
])

# b (Answers) = 0 for Fail, 1 for Pass
b = np.array([0, 0, 0, 0, 1, 1, 1, 1])


model = DecisionTreeClassifier()
score = cross_val_score(model,a,b,cv=4)
print(score.mean())

1.0


### The Professional ML Pipeline: Preventing Data Leakage

Imagine you are a teacher, and your job is to prepare a student for their **Final SAT Exam**. You have a big book of 100 math problems.

#### The Problem: Memorizing the Answers
If you use all 100 math problems to teach the student, and then you use those exact same 100 problems for the Final Exam... the student will score a 100%. 

But did they actually learn math? **No.** They just memorized that the answer to Question 4 is "C". 

If you send them out into the real world, they will fail immediately. This is the biggest danger in Machine Learning. We call it **"Data Leakage"** or **"Overfitting."** The computer acts like a lazy student and just memorizes the data instead of finding the real pattern.

---

#### The Professional Solution: The Two-Step Test
To stop the computer from cheating, professional data scientists use two different security measures at the exact same time.

**Step 1: The Vault (`train_test_split`)**
Before you even start teaching, you rip 20 pages out of the math book and lock them in a vault. **You absolutely do not let the student see them.** These 20 pages are the True Final Exam.

```python
# SYNTAX: Locking 20% of the data in the vault
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
```

**Step 2: The Practice Tests (`cross_val_score`)**
You only have 80 pages left (`X_train` and `y_train`). You divide these 80 pages into 4 smaller "Practice Exams." You use these practice scores to figure out if the student is actually learning the math concepts or if they need to study harder.

```python
# SYNTAX: Running 4 Practice Exams on the remaining 80% (X_train)
practice_scores = cross_val_score(model, X_train, y_train, cv=4)
print(f"Practice Average: {practice_scores.mean() * 100}%")
```

**Step 3: The Real World Test**
Once the student gets a high average on all their Practice Exams, you say, *"Okay, you are ready."* You officially teach them the 80 pages. Then, you finally open the vault, pull out those 20 hidden pages (`X_test`), and give them the True Final Exam. 

If they score high on the vault data, you know for a fact they didn't cheat, because they had never seen those questions before!

```python
# SYNTAX: Official teaching session
model.fit(X_train, y_train)

# Unlock the vault and test!
final_predictions = model.predict(X_test)
final_grade = accuracy_score(y_test, final_predictions)
print(f"Real World Accuracy: {final_grade * 100}%")
```

In [19]:
# s (Clues) = [Hours Studied, Classes Attended] -> Notice the 2 columns!
s = np.array([
    [1, 2], [2, 3], [3, 4], [4, 2],  # These students studied little & skipped class
    [6, 8], [7, 9], [8, 8], [9, 10]  # These students studied a lot & attended class
])

# r (Answers) = 0 for Fail, 1 for Pass
r = np.array([0, 0, 0, 0, 1, 1, 1, 1])

X_train, X_test, y_train, y_test = train_test_split(
    s, r, 
    test_size=0.25, 
    random_state=42, 
    stratify=r  # <--- This locks the ratio!
)

# 3. PROOF OF THE RATIO
print("--- Original Data ---")
print(f"Values in r: {r}")

print("\n--- The 'Vault' (Test Set) ---")
print(f"Values in y_test: {y_test}") 
# Notice it contains exactly ONE '0' and ONE '1' (50/50)

print("\n--- The Training Set ---")
print(f"Values in y_train: {y_train}")
# Notice it contains exactly THREE '0's and THREE '1's (50/50)

--- Original Data ---
Values in r: [0 0 0 0 1 1 1 1]

--- The 'Vault' (Test Set) ---
Values in y_test: [0 1]

--- The Training Set ---
Values in y_train: [1 0 1 1 0 0]
